## Column Transformer (in Machine Learning)

- A Column Transformer is a tool in scikit-learn that allows you to apply different preprocessing steps to different columns of a dataset at the same time.

- This is very useful when your dataset contains different types of features such as:

  -  Numerical columns
    
  -  Categorical columns
    
  -  Text columns

- Each type requires different preprocessing techniques.
    

- Different columns need different preprocessing:
    | Column Type  | Preprocessing |
| ------------ | ------------- |
| Age, Salary  | Scaling       |
| Gender, City | Encoding      |
        

- Without ColumnTransformer, you must preprocess them separately.

- With ColumnTransformer, you can handle everything in one pipeline.


### Advantages

    ✔ Handles mixed data types
        
    ✔ Clean and organized preprocessing
        
    ✔ Works well with pipelines
        
    ✔ Prevents data leakage

# Column Tranformer

In [3]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/main/day28-column-transformer/covid_toy.csv"

df = pd.read_csv(url)

df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
df.shape

(100, 6)

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder # work on nominal catogirical data
from sklearn.preprocessing import OrdinalEncoder # work on ordinal independent categorical data

In [6]:
df.isna().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
4,65,Female,101.0,Mild,Mumbai
72,83,Female,101.0,Mild,Kolkata
47,18,Female,104.0,Mild,Bangalore
96,51,Female,101.0,Strong,Kolkata
46,19,Female,101.0,Mild,Mumbai
...,...,...,...,...,...
75,5,Male,102.0,Mild,Kolkata
89,46,Male,103.0,Strong,Bangalore
29,34,Female,NaN,Strong,Mumbai
50,19,Male,101.0,Mild,Delhi


# 1. Aam Zindagi

In [11]:
# adding simple imputer to fevel col
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[['fever']])

In [12]:
# also for test data
X_test_fever=si.transform(X_test[['fever']])

In [13]:
X_test_fever.shape

(20, 1)

In [16]:
X_train_fever.shape

(80, 1)

In [18]:
# Ordinalencoding -> cough
oe=OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough=oe.fit_transform(X_train[['cough']])
# also for test data
X_test_cough=oe.transform(X_test[['cough']])

In [19]:
X_train_cough.shape

(80, 1)

In [23]:
# OneHotEncoding->gender,city
ohe=OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city=ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city=ohe.transform(X_test[['gender','city']])

In [25]:
X_train_gender_city.shape

(80, 4)

In [30]:
# Extracting Age
X_train_age=X_train.drop(columns=['gender','fever','cough','city']).values
# also for test data
X_test_age=X_train.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape,X_test.shape

((80, 1), (20, 5))

In [29]:
# Merging all the data which scaled and encoded

X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 80 and the array at index 1 has size 20

# Mentos Zindagi

In [31]:
from sklearn.compose import ColumnTransformer

In [34]:
transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [39]:
X_train_col_trans=transformer.fit_transform(X_train)

In [40]:
X_test_col_trans=transformer.transform(X_test)